In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/test_bench/checkpoints/pre_cell_3.pickle

In [4]:
%%RecordEvent
%%time
### cell 3 ###

def combine_subset_of_data_from_multiple_years(question_of_interest, x_axis_title, include_2017=None):
    years = [2022, 2021, 2020, 2019, 2018]
    if include_2017 is not None:
        years.append(2017)
    responses_map = {
        2022: responses_df_2022,
        2021: responses_df_2021,
        2020: responses_df_2020,
        2019: responses_df_2019,
        2018: responses_df_2018,
        2017: responses_df_2017,
    }
    # Concatenate all years' question responses into one DataFrame
    sorted_years = sorted(years)
    df_all = pd.concat(
        [
            responses_map[year][[question_of_interest]].assign(year=year)
            for year in sorted_years
        ],
        ignore_index=True
    )
    # Compute counts per category and year, then percentages
    df_counts = (
        df_all
        .groupby(['year', question_of_interest])
        .size()
        .reset_index(name='count')
    )
    total_per_year = df_counts.groupby('year')['count'].transform('sum')
    df_counts['percentage'] = df_counts['count'] / total_per_year * 100
    # Rename and reorder to match original output
    df_result = (
        df_counts
        .rename(columns={question_of_interest: x_axis_title})
        [['percentage', 'year', x_axis_title]]
    )
    return df_result


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions(question_of_interest, include_2017=None):
    years = [2022, 2021, 2020, 2019, 2018]
    if include_2017 is not None:
        years.append(2017)
    responses_map = {
        2022: responses_df_2022,
        2021: responses_df_2021,
        2020: responses_df_2020,
        2019: responses_df_2019,
        2018: responses_df_2018,
        2017: responses_df_2017,
    }
    sorted_years = sorted(years)
    # Build combined DataFrame in one go
    df_list = [
        grab_subset_of_data(responses_map[year], question_of_interest).assign(year=year)
        for year in sorted_years
    ]
    df_combined = pd.concat(df_list, ignore_index=True)
    # Compute per-year counts
    df_combined_counts = df_combined.groupby('year').count().reset_index()
    return df_combined, df_combined_counts


def load_survey_data(base_dir, file_name, rows_to_skip=1):
    file_path = os.path.join(base_dir, file_name)
    df = pd.read_csv(file_path, low_memory=False, encoding='ISO-8859-1', skiprows=rows_to_skip)
    out_name = f"kaggle_survey_{base_dir[-5:-1]}.csv"
    if not glob.glob(out_name):
        df.to_csv(out_name, index=False)
    return df

Error parsing UDF grab_subset_of_data: could not get source code
CPU times: user 4 µs, sys: 1 µs, total: 5 µs
Wall time: 8.11 µs


In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small/checkpoints/post_cell_3_try_1.pickle